# 🏦 I&M Bank Integrated Analytics Engine
## Multi-Dimensional Customer & Portfolio Analysis
### 5,019 Transactions | 600 Customers | 250 Loans | 6 Branches | Full Year 2025

In [1]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
plt.rcParams['figure.dpi'] = 130
plt.rcParams['font.family'] = 'DejaVu Sans'
sns.set_theme(style='whitegrid')

print('Environment initialized | I&M Bank Analytics Engine')

Environment initialized | I&M Bank Analytics Engine


In [2]:
# Load from Excel
from pathlib import Path

workbook_name = 'IM_Bank_Practice_Data.xlsx'
file_candidates = [
    Path.cwd() / workbook_name,
    Path.cwd() / 'notebooks:data' / workbook_name,
    Path.cwd().parent / 'notebooks:data' / workbook_name,
]
file_path = next((path for path in file_candidates if path.exists()), None)
if file_path is None:
    raise FileNotFoundError(f'Could not find {workbook_name}. Checked: {file_candidates}')

print(f'Loading workbook: {file_path}')
txn_raw = pd.read_excel(file_path, sheet_name='Transactions')
cust = pd.read_excel(file_path, sheet_name='Customers')
loans = pd.read_excel(file_path, sheet_name='Loan_Book')

# Clean transactions
txn = txn_raw[['Date', 'Branch', 'Channel', 'Transaction Type', 'Account Type', 'Customer ID', 'Amount (KES)', 'Status']].copy()
txn = txn.dropna(subset=['Date', 'Branch', 'Channel'])
txn.columns = ['Date', 'Branch', 'Channel', 'Transaction_Type', 'Account_Type', 'Customer_ID', 'Amount', 'Status']
txn['Date'] = pd.to_datetime(txn['Date'])

# Clean customers
cust.columns = ['Customer_ID', 'Segment', 'Region', 'Onboarding_Channel', 'Date_Joined', 'Active']
cust['Active'] = cust['Active'].map({'Yes': 1, 'No': 0})

# Clean loans
loans.columns = ['Loan_ID', 'Customer_ID', 'Branch', 'Loan_Type', 'Principal', 'Interest_Rate', 'Term_Months', 'Disbursed_Date', 'Status']

print(f'Transactions: {len(txn):,} rows | Date range: {txn["Date"].min().date()} to {txn["Date"].max().date()}')
print(f'Customers: {len(cust):,} rows | Active: {cust["Active"].sum():,}')
print(f'Loans: {len(loans):,} rows | Status: {loans["Status"].value_counts().to_dict()}')
print(f'Total transaction value: KES {txn["Amount"].sum():,.0f}')
print(f'Total loan principal: KES {loans["Principal"].sum():,.0f}')

Loading workbook: /Users/user/Downloads/i&mbankanalysis/notebooks:data/IM_Bank_Practice_Data.xlsx
Transactions: 5,019 rows | Date range: 2025-01-01 to 2025-12-31
Customers: 600 rows | Active: 553
Loans: 250 rows | Status: {'Performing': 202, 'Watch': 32, 'Non-Performing': 16}
Total transaction value: KES 245,836,908
Total loan principal: KES 528,505,354


In [3]:
# Transaction analysis
txn['Month'] = txn['Date'].dt.to_period('M')
txn['Week'] = txn['Date'].dt.isocalendar().week
txn['Day_of_Week'] = txn['Date'].dt.day_name()
txn['Success'] = (txn['Status'] == 'Completed').astype(int)

# Customer lifetime value
cust_txn_value = txn.groupby('Customer_ID').agg({'Amount': 'sum', 'Customer_ID': 'count'}).rename(columns={'Amount': 'Total_Txn_Value', 'Customer_ID': 'Txn_Count'}).reset_index()
cust_txn_value['Avg_Txn_Size'] = cust_txn_value['Total_Txn_Value'] / cust_txn_value['Txn_Count']

# Merge with customer data
cust_full = cust.merge(cust_txn_value, on='Customer_ID', how='left').fillna(0)

# Loan metrics
loans['Monthly_Payment'] = loans['Principal'] / loans['Term_Months']
loans['Annual_Interest'] = loans['Principal'] * loans['Interest_Rate']
loans['NPL_Flag'] = (loans['Status'] == 'Non-Performing').astype(int)

# Merge loans with customers
loans_cust = loans.merge(cust[['Customer_ID', 'Segment', 'Region']], on='Customer_ID', how='left')

# Cross-sell analysis
cust_with_loans = set(loans['Customer_ID'].unique())
cust_active = set(cust[cust['Active'] == 1]['Customer_ID'].unique())
cust_txn = set(txn['Customer_ID'].unique())

cust_full['Has_Loan'] = cust_full['Customer_ID'].isin(cust_with_loans).astype(int)
cust_full['Has_Txn'] = cust_full['Customer_ID'].isin(cust_txn).astype(int)
cust_full['Txn_but_No_Loan'] = ((cust_full['Has_Txn'] == 1) & (cust_full['Has_Loan'] == 0)).astype(int)
cust_full['Loan_but_No_Txn'] = ((cust_full['Has_Loan'] == 1) & (cust_full['Has_Txn'] == 0)).astype(int)

print(f'Feature engineering complete')
print(f'Customers with loans: {cust_full["Has_Loan"].sum():,} ({cust_full["Has_Loan"].mean()*100:.1f}%)')
print(f'Customers with transactions: {cust_full["Has_Txn"].sum():,} ({cust_full["Has_Txn"].mean()*100:.1f}%)')
print(f'Cross-sell opportunity (txn but no loan): {cust_full["Txn_but_No_Loan"].sum():,}')
print(f'Portfolio NPL rate: {loans["NPL_Flag"].mean()*100:.2f}%')

Feature engineering complete
Customers with loans: 209 (34.8%)
Customers with transactions: 600 (100.0%)
Cross-sell opportunity (txn but no loan): 391
Portfolio NPL rate: 6.40%
